# Solar Filament Segmentation - two models, two GPUs, one session

Trains **two architectures concurrently**, one per GPU, on Kaggle's *T4 x2*
accelerator. A single training run only ever uses `cuda:0`, so the second card
is normally idle for the whole session; this fills it. Two models finish in
roughly the wall-clock time of one.

Set the accelerator to **GPU T4 x2** (not P100 - that is a single GPU) under
Settings, and enable **Internet**.

Like the other runner notebook, this holds no logic: it clones the repo at a
pinned revision and calls `scripts/train_parallel.py`.

In [ ]:
# --- the only cell you normally edit -----------------------------------------
REPO_URL = "https://github.com/ShreyPatel1311/solar-filament-segmentation.git"
REVISION = "main"          # branch, tag, or full commit SHA

CONFIGS = [                # one per GPU
    "configs/flat_unet.yaml",       # Zhu et al. 2025, ~0.3M params
    "configs/diercke_unet.yaml",    # Diercke et al. 2024, ~31M params
]
OVERRIDES = []             # applied to BOTH runs, e.g. ["train.epochs=30"]
HF_REPO_ID = None          # e.g. "you/filament-models" to push both checkpoints
# -----------------------------------------------------------------------------

In [ ]:
import os, subprocess, sys, shutil, pathlib

WORK_DIR = pathlib.Path("/kaggle/working")
os.chdir(WORK_DIR)  # stand outside REPO_DIR before deleting it, so re-running
                    # this cell without a kernel restart cannot strand the cwd

REPO_DIR = WORK_DIR / "repo"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REVISION], check=True)

commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                        check=True, capture_output=True, text=True).stdout.strip()
print("running commit", commit)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
os.environ["PYTHONPATH"] = str(REPO_DIR / "src")

In [ ]:
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

!pip install -q -r requirements-kaggle.txt
!pip install -q --no-deps -r requirements-kaggle-nodeps.txt

In [ ]:
# Confirm there really are two GPUs before launching -- with one visible GPU
# both runs would land on the same card and likely run out of memory.
import torch

print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
print("visible GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  cuda:{i}  {name}  {total:.1f} GB")

if torch.cuda.device_count() < 2:
    print("\n!! Fewer than 2 GPUs. Set Accelerator to 'GPU T4 x2' and restart,")
    print("   or train one config at a time with the standard runner notebook.")

## Train both

Output is interleaved and tagged with each config's `name:`, so
`[flat_unet]` and `[diercke_unet]` lines appear as they happen. Each run
writes its own `{name}_best.pt`, `{name}_last.pt` and `{name}_history.json`
to `/kaggle/working/checkpoints` - the launcher refuses to start if two
configs share a `name:`, since they would overwrite each other.

A crash in one run does not touch the other; the summary at the end reports
each run's status separately.

In [ ]:
args = " ".join(CONFIGS)
args += "".join(f" --set {o}" for o in OVERRIDES)
if HF_REPO_ID:
    args += f" --hf-repo-id {HF_REPO_ID}"

!python scripts/train_parallel.py {args}

## Compare the two runs

In [ ]:
import json, pathlib
import pandas as pd

from filseg.paths import resolve_paths

paths = resolve_paths()
rows = []
for history_file in sorted(paths.checkpoints.glob("*_history.json")):
    history = json.loads(history_file.read_text())
    best = max(history, key=lambda r: r["val_dice"])
    rows.append({
        "model": history_file.stem.replace("_history", ""),
        "epochs": len(history),
        "best_val_dice": round(best["val_dice"], 4),
        "best_epoch": best["epoch"],
        "final_train_loss": round(history[-1]["train_loss"], 4),
        "min/epoch": round(sum(r["seconds"] for r in history) / len(history) / 60, 1),
    })

pd.DataFrame(rows).sort_values("best_val_dice", ascending=False)

## Score both with the leaderboard metric

Validation Dice above is only a proxy. Panoptic Quality is what the
leaderboard uses, and it penalises fragmentation and over-merging that Dice
is blind to - so rank the two models on PQ, not Dice.

In [ ]:
for config in CONFIGS:
    name = __import__("yaml").safe_load(open(config))["name"]
    checkpoint = paths.checkpoints / f"{name}_best.pt"
    if checkpoint.exists():
        print(f"\n{'=' * 70}\n{name}\n{'=' * 70}")
        !python scripts/evaluate.py --checkpoint {checkpoint}